# Demo: refinamiento y condiciones de parada (sin API)

Complementa a [`demo.ipynb`](demo.ipynb). Acá el grafo, el estado, las reglas duras y el validador son **los reales**; lo único que se reemplaza por dobles guionados ([`tests/doubles.py`](tests/doubles.py)) son el LLM del supervisor, los dos agentes ReAct y el sintetizador. Así se fuerzan, de forma reproducible, situaciones que con un LLM real aparecen solo a veces. Corre sin clave de API.

Una corrida **real** en la que el supervisor pidió refinar al Investigador quedó en [`docs/corrida_refinamiento.txt`](docs/corrida_refinamiento.txt).

In [1]:
from graph import build_graph
from main import run_with_trace
from tests.doubles import (GOOD_ANSWER, INVENTED_ANSWER, RESEARCH_ONLY_ANSWER, analyst_double,
                           research_double, scripted_decider, scripted_llm)

QUESTION = '¿Cuánta CPU reserva pagos-api en su máximo de réplicas?'

def ruta(result):
    return ' → '.join(d['route'] for d in result.state['decisions'])

## 1. Refinar al Investigador y corregir un número inventado

El supervisor guionado rechaza la primera investigación (no trae el máximo de réplicas) y después intenta cerrar dos veces. El sintetizador guionado escribe primero `7 cores`, un número que ninguna herramienta produjo: el validador lo rechaza y la regla dura no deja cerrar hasta que se corrige.

In [2]:
decide, _ = scripted_decider([
    ('researcher', 'Buscá cpu y memoria de pagos-api.'),
    ('researcher', 'Falta el máximo de réplicas de pagos-api: buscalo.'),
    ('analyst', 'Calculá la CPU total: 10 réplicas x 0.5 cores.'),
    ('synthesizer', 'Redactá la respuesta.'),
    ('FINISH', ''),  # quiere cerrar con la validación fallida -> regla dura
    ('FINISH', ''),
])
research, _ = research_double(partial_first=True)  # su primer aporte omite el máximo de réplicas
analyst, _ = analyst_double()
synthesizer, _ = scripted_llm(INVENTED_ANSWER, GOOD_ANSWER)
graph = build_graph(decide=decide, research_agent=research, analyst_agent=analyst, synthesizer_llm=synthesizer)
r = run_with_trace(graph, QUESTION)
print('Ruta:', ruta(r))

🧭 supervisor · decisión 1 → researcher
   evaluación: (guion)
   motivo: (guion)
   instrucción: Buscá cpu y memoria de pagos-api.

🔎 researcher · intento 1 · herramientas: search_docs×1 · fragmentos: 02_despliegue_y_cli.md#4
   │ Hallazgos:
   │ - pagos-api pide cpu 500m y memoria 512Mi [02_despliegue_y_cli.md#4]
   │ No encontrado:
   │ - el máximo de réplicas

🧭 supervisor · decisión 2 → researcher
   evaluación: (guion)
   motivo: (guion)
   instrucción: Falta el máximo de réplicas de pagos-api: buscalo.

🔎 researcher · intento 2 · herramientas: search_docs×1 · fragmentos: 02_despliegue_y_cli.md#4
   │ Hallazgos:
   │ - pagos-api pide cpu 500m y memoria 512Mi, con replicas.max 10 [02_despliegue_y_cli.md#4]
   │ No encontrado:
   │ - nada

🧭 supervisor · decisión 3 → analyst
   evaluación: (guion)
   motivo: (guion)
   instrucción: Calculá la CPU total: 10 réplicas x 0.5 cores.

🧮 analyst · intento 1 · herramientas: calculate×1
   │ Cálculos:
   │ - CPU total: 10 * 0.5 = 5 cores (da

## 2. El "supervisor infinito"

Ahora el LLM del supervisor pide *más investigación* en cada turno, para siempre. Las reglas duras lo cortan: el Investigador tiene 2 intentos, después se sintetiza con lo disponible y se cierra.

In [3]:
decide, _ = scripted_decider([('researcher', 'Buscá un poco más.')] * 50)
research, _ = research_double()
analyst, _ = analyst_double()
synthesizer, _ = scripted_llm(RESEARCH_ONLY_ANSWER)
graph = build_graph(decide=decide, research_agent=research, analyst_agent=analyst, synthesizer_llm=synthesizer)
r = run_with_trace(graph, QUESTION)
print(f"{len(r.state['decisions'])} decisiones · ruta:", ruta(r))

🧭 supervisor · decisión 1 → researcher
   evaluación: (guion)
   motivo: (guion)
   instrucción: Buscá un poco más.

🔎 researcher · intento 1 · herramientas: search_docs×1 · fragmentos: 02_despliegue_y_cli.md#4
   │ Hallazgos:
   │ - pagos-api pide cpu 500m y memoria 512Mi, con replicas.max 10 [02_despliegue_y_cli.md#4]
   │ No encontrado:
   │ - nada

🧭 supervisor · decisión 2 → researcher
   evaluación: (guion)
   motivo: (guion)
   instrucción: Buscá un poco más.

🔎 researcher · intento 2 · herramientas: search_docs×1 · fragmentos: 02_despliegue_y_cli.md#4
   │ Hallazgos:
   │ - pagos-api pide cpu 500m y memoria 512Mi, con replicas.max 10 [02_despliegue_y_cli.md#4]
   │ No encontrado:
   │ - nada

🧭 supervisor · decisión 3 → synthesizer
   evaluación: (guion)
   motivo: (guion)
   ⛔ regla dura: researcher ya usó sus 2 intentos: se sigue con synthesizer.
   instrucción: Redactá la respuesta final con los aportes disponibles y aclará qué no se pudo resolver.

✍️  synthesizer · intento

## 3. Un aporte que llega después del borrador

Con un borrador ya validado, el supervisor pide un cálculo más y enseguida quiere cerrar. La regla dura detecta que el borrador no incorpora ese aporte y pide actualizarlo antes del `END`.

In [4]:
decide, _ = scripted_decider([
    ('researcher', 'Buscá cpu, memoria y réplicas de pagos-api.'),
    ('synthesizer', 'Redactá la respuesta.'),
    ('analyst', 'Calculá la CPU total: 10 réplicas x 0.5 cores.'),
    ('FINISH', ''),  # el borrador no incluye el cálculo -> regla dura
    ('FINISH', ''),
])
research, _ = research_double()
analyst, _ = analyst_double()
synthesizer, _ = scripted_llm(RESEARCH_ONLY_ANSWER, GOOD_ANSWER)
graph = build_graph(decide=decide, research_agent=research, analyst_agent=analyst, synthesizer_llm=synthesizer)
r = run_with_trace(graph, QUESTION)
print('Ruta:', ruta(r))
print('Respuesta final:', r.state['final_answer'])

🧭 supervisor · decisión 1 → researcher
   evaluación: (guion)
   motivo: (guion)
   instrucción: Buscá cpu, memoria y réplicas de pagos-api.

🔎 researcher · intento 1 · herramientas: search_docs×1 · fragmentos: 02_despliegue_y_cli.md#4
   │ Hallazgos:
   │ - pagos-api pide cpu 500m y memoria 512Mi, con replicas.max 10 [02_despliegue_y_cli.md#4]
   │ No encontrado:
   │ - nada

🧭 supervisor · decisión 2 → synthesizer
   evaluación: (guion)
   motivo: (guion)
   instrucción: Redactá la respuesta.

✍️  synthesizer · intento 1
   │ pagos-api pide 500m de CPU y 512Mi de memoria por réplica, con un máximo de 10 réplicas [02_despliegue_y_cli.md#4].

🛡️  validator · APROBADA: 3 números con respaldo · citas válidas: 1

🧭 supervisor · decisión 3 → analyst
   evaluación: (guion)
   motivo: (guion)
   instrucción: Calculá la CPU total: 10 réplicas x 0.5 cores.

🧮 analyst · intento 1 · herramientas: calculate×1
   │ Cálculos:
   │ - CPU total: 10 * 0.5 = 5 cores (datos: 10 réplicas, 500m)
   │ Dato